# SQL Database — MovieRecommenderDB

This notebook prepares and loads the MovieLens 25M data into the SQL Server database used by the recommendation system.

The main steps are:

- Connect to SQL Server
- Prepare the MovieLens data for the relational schema
- Create user, movie, genre, and movie-genre datasets
- Load the prepared data into SQL Server
- Validate the loaded data before inserting ratings
- Load the 25M ratings in batches

## Imports

In [ ]:
from sqlalchemy import create_engine
import pandas as pd

## Database Connection

The recommendation system uses SQL Server to store structured movie and user-rating data.

The database used in this project is `MovieRecommenderDB`.

In [ ]:
server = "."
database = "MovieRecommenderDB"

connection_string = (
    f"mssql+pyodbc://{server}/{database}"
    "?trusted_connection=yes"
    "&driver=ODBC+Driver+17+for+SQL+Server"
)

In [ ]:
engine = create_engine(connection_string)

In [ ]:
with engine.connect() as connection:
    print("Database connected successfully")

## Load MovieLens Data

The original MovieLens 25M CSV files are loaded into pandas before being transformed into the structure required by the SQL database.

The main datasets are:

- `movies.csv`
- `ratings.csv`

In [ ]:
movies = pd.read_csv("../data/raw/ml-25m/movies.csv")
ratings = pd.read_csv("../data/raw/ml-25m/ratings.csv")

## Prepare Data for the SQL Schema

The raw MovieLens data is transformed into separate datasets corresponding to the relational database schema.

The main entities are:

- Users
- Movies
- Genres
- MovieGenres
- Ratings

In [ ]:
users = pd.DataFrame({"user_id": ratings["userId"].unique()})
movies_sql = movies[["movieId", "title"]].copy()
ratings_sql = ratings[["userId", "movieId", "rating"]].copy()

### Genre Data

Movie genres are stored separately so that each genre can be represented by a unique identifier.

In [ ]:
genres = (movies["genres"].str.split("|").explode().drop_duplicates().sort_values().reset_index(drop=True))
genres = pd.DataFrame({"genre_id" : range(1, len(genres) + 1), "genre_name": genres})

### Align Column Names with the SQL Schema

In [ ]:
movie_genres = (movies[["movieId", "genres"]].assign(genre_name= movies["genres"].str.split("|")).explode("genre_name").drop(columns="genres"))
movie_genres = movie_genres.merge(genres, on="genre_name", how="left")
movie_genres = movie_genres[["movieId", "genre_id"]]

In [ ]:
movies_sql = movies_sql.rename(columns={"movieId" : "movie_id"})
ratings_sql = ratings_sql.rename(columns={"movieId" : "movie_id", "userId" : "user_id"})
movie_genres = movie_genres.rename(columns={"movieId" : "movie_id"})

## Load Data into SQL Server

The prepared datasets are inserted into their corresponding SQL Server tables.

The parent tables are populated before the relationship and rating tables.

In [ ]:
users.to_sql("Users", con=engine, if_exists="append", index=False)
movies_sql.to_sql("Movies", con=engine, if_exists="append", index=False)

In [ ]:
genres_to_sql = genres["genre_name"].copy()
genres_to_sql.to_sql("Genres", con=engine, if_exists="append", index=False)

In [ ]:
movie_genres.to_sql("MovieGenres", con=engine, if_exists="append", index=False)

## Data Validation

Before loading the ratings, the movie–genre relationships are checked for missing genre identifiers and duplicate movie–genre pairs.

In [ ]:
print(movie_genres.head())

print(
    "Missing genre IDs:",
    movie_genres["genre_id"].isna().sum()
)

print(
    "Duplicate movie-genre pairs:",
    movie_genres.duplicated(
        subset=["movie_id", "genre_id"]
    ).sum()
)

## Load Ratings in Batches

The MovieLens 25M dataset contains millions of rating records, so the `Ratings` table is populated in batches rather than inserting all records at once.

The first batch is inserted separately, followed by the remaining ratings using larger chunks.

In [ ]:
ratings_sample = ratings_sql.head(1000)

ratings_sample.to_sql(
    "Ratings",
    con=engine,
    if_exists="append",
    index=False,
    chunksize=1000
)

In [ ]:
ratings_remaining = ratings_sql.iloc[1000:]

ratings_remaining.to_sql(
    "Ratings",
    con=engine,
    if_exists="append",
    index=False,
    chunksize=10000
)

## Database Preparation Complete

The MovieLens data has been prepared and loaded into the SQL Server database.

The resulting relational structure provides the data required for the recommendation system, including:

- User–movie ratings for collaborative filtering
- Movie metadata for content-based recommendation
- Movie–genre relationships for genre-based features

The database is now ready for the subsequent feature engineering and recommendation-modeling steps.

> **Note:** This notebook uses `if_exists="append"` when inserting data. It is intended for initial database population and should not be re-run on an already populated database without resetting or clearing the relevant tables first.